In [2]:
"""
Word2Vec Tutorial and Examples

This file demonstrates how to use the Word2Vec implementation from scratch.
Follow along to understand each concept!
"""

from word2vec_scratch import Word2VecSkipGram
import numpy as np


In [3]:

# ============================================================================
# PART 1: BASIC CONCEPT EXPLANATION
# ============================================================================
"""
What is Word2Vec?
-----------------
Word2Vec is an algorithm that learns dense vector representations of words.

Why do we need embeddings?
- Words are discrete symbols (cat, dog, run, etc.)
- Machine learning models work better with continuous numbers
- We want similar words to have similar vectors
  Example: embedding(dog) should be similar to embedding(puppy)

How does Skip-gram work?
1. Takes a target word (e.g., "dog")
2. Looks at words near it in context (e.g., "the", "barks", "loudly")
3. Learns to maximize similarity between target and context word embeddings
4. This naturally groups semantically related words together

Key insight: "Words are known by the company they keep" (distributional hypothesis)
If "dog" and "puppy" appear in similar contexts, their embeddings will be similar!
"""


'\nWhat is Word2Vec?\n-----------------\nWord2Vec is an algorithm that learns dense vector representations of words.\n\nWhy do we need embeddings?\n- Words are discrete symbols (cat, dog, run, etc.)\n- Machine learning models work better with continuous numbers\n- We want similar words to have similar vectors\n  Example: embedding(dog) should be similar to embedding(puppy)\n\nHow does Skip-gram work?\n1. Takes a target word (e.g., "dog")\n2. Looks at words near it in context (e.g., "the", "barks", "loudly")\n3. Learns to maximize similarity between target and context word embeddings\n4. This naturally groups semantically related words together\n\nKey insight: "Words are known by the company they keep" (distributional hypothesis)\nIf "dog" and "puppy" appear in similar contexts, their embeddings will be similar!\n'

In [4]:
# ============================================================================
# PART 2: PREPARE SIMPLE DATA
# ============================================================================

# Simple example with short sentences
simple_sentences = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are animals",
    "I like to play with my cat",
    "I like to play with my dog",
    "the mat is soft and warm",
    "dogs are loyal and friendly",
    "cats are independent and clever",
]

# Tokenize sentences
def tokenize_simple(text):
    """Simple tokenization: split by spaces and lowercase."""
    return text.lower().split()

sentences = [tokenize_simple(sent) for sent in simple_sentences]

print("=" * 70)
print("PART 2: DATA PREPARATION")
print("=" * 70)
print("\nSample sentences (tokenized):")
for i, sent in enumerate(sentences[:3]):
    print(f"  {i+1}. {sent}")


PART 2: DATA PREPARATION

Sample sentences (tokenized):
  1. ['the', 'cat', 'sat', 'on', 'the', 'mat']
  2. ['the', 'dog', 'sat', 'on', 'the', 'log']
  3. ['cats', 'and', 'dogs', 'are', 'animals']


In [5]:
# ============================================================================
# PART 3: TRAIN WORD2VEC MODEL
# ============================================================================

print("\n" + "=" * 70)
print("PART 3: TRAINING WORD2VEC MODEL")
print("=" * 70)
print("\nCreating Word2Vec model with:")
print("  - Embedding dimension: 50")
print("  - Window size: 2 (look 2 words left and right)")
print("  - Negative samples: 5")
print("  - Learning rate: 0.025")

model = Word2VecSkipGram(
    embedding_dim=50,
    window_size=2,
    negative_samples=5,
    learning_rate=0.025,
    min_count=1
)

print("\nStarting training...")
model.train(sentences, epochs=100, verbose=False)

# Print final stats
print(f"\nTraining complete!")
print(f"Vocabulary contains {model.vocab_size} unique words (after filtering)")



PART 3: TRAINING WORD2VEC MODEL

Creating Word2Vec model with:
  - Embedding dimension: 50
  - Window size: 2 (look 2 words left and right)
  - Negative samples: 5
  - Learning rate: 0.025

Starting training...
Vocabulary size: 25
Negative sampling table size: 999987

Training complete! Total pairs trained: 10925

Training complete!
Vocabulary contains 25 unique words (after filtering)


In [6]:

# ============================================================================
# PART 4: EXPLORING EMBEDDINGS
# ============================================================================

print("\n" + "=" * 70)
print("PART 4: EXPLORING WORD EMBEDDINGS")
print("=" * 70)

# Get embedding for a specific word
test_word = "dog"
vec = model.get_vector(test_word)
print(f"\nEmbedding for '{test_word}':")
print(f"  Shape: {vec.shape}")
print(f"  First 10 dimensions: {vec[:10]}")
print(f"  L2 norm (length): {np.linalg.norm(vec):.4f}")

# ============================================================================
# PART 5: FINDING SIMILAR WORDS
# ============================================================================

print("\n" + "=" * 70)
print("PART 5: FINDING SIMILAR WORDS")
print("=" * 70)
print("\nThis demonstrates the key property of word embeddings:")
print("Words appearing in similar contexts have similar vectors!\n")

test_words = ["dog", "cat", "play", "soft"]

for word in test_words:
    try:
        similar = model.most_similar(word, topn=3)
        print(f"Most similar to '{word}':")
        for similar_word, score in similar:
            print(f"  - {similar_word}: {score:.4f}")
    except ValueError as e:
        print(f"Word '{word}' not in vocabulary!")


PART 4: EXPLORING WORD EMBEDDINGS

Embedding for 'dog':
  Shape: (50,)
  First 10 dimensions: [ 0.07001765 -0.00119952  0.02668949  0.02113028  0.00360125  0.00558609
  0.00276654  0.00899526 -0.03903237 -0.02044035]
  L2 norm (length): 0.1943

PART 5: FINDING SIMILAR WORDS

This demonstrates the key property of word embeddings:
Words appearing in similar contexts have similar vectors!

Most similar to 'dog':
  - cat: 0.8801
  - on: 0.8704
  - sat: 0.8403
Most similar to 'cat':
  - dog: 0.8801
  - on: 0.8766
  - sat: 0.8572
Most similar to 'play':
  - to: 0.9385
  - with: 0.9290
  - my: 0.8668
Most similar to 'soft':
  - and: 0.9177
  - is: 0.8505
  - are: 0.8490


In [7]:
# ============================================================================
# PART 6: WORD ANALOGIES (ADVANCED)
# ============================================================================

print("\n" + "=" * 70)
print("PART 6: WORD ANALOGIES")
print("=" * 70)
print("""
Word analogies work using vector arithmetic!

Example: king - man + woman ≈ queen

This works because embeddings capture semantic relationships:
  king - man captures the concept of "royalty"
  Adding woman captures "female"
  So king - man + woman ≈ female ruler ≈ queen

In our small dataset, let's try: dog - animals + pets
(We'll see what words are similar to this vector combination)
""")

try:
    # Try an analogy
    analogy_result = model.analogy(
        positive_words=["dog", "play"],
        negative_words=["cat"],
        topn=3
    )
    print("Words similar to [dog + play - cat]:")
    for word, score in analogy_result:
        print(f"  - {word}: {score:.4f}")
except Exception as e:
    print(f"Analogy example: {e}")


PART 6: WORD ANALOGIES

Word analogies work using vector arithmetic!

Example: king - man + woman ≈ queen

This works because embeddings capture semantic relationships:
  king - man captures the concept of "royalty" 
  Adding woman captures "female"
  So king - man + woman ≈ female ruler ≈ queen

In our small dataset, let's try: dog - animals + pets
(We'll see what words are similar to this vector combination)

Words similar to [dog + play - cat]:
  - with: 0.9198
  - to: 0.9082
  - like: 0.8412


In [8]:
# ============================================================================
# PART 7: VISUALIZING WHAT THE MODEL LEARNED
# ============================================================================

print("\n" + "=" * 70)
print("PART 7: UNDERSTANDING EMBEDDINGS")
print("=" * 70)

print("\nSample vocabulary learned:")
vocab_words = list(model.word2idx.keys())
print(f"Total words: {len(vocab_words)}")
print(f"Sample words: {vocab_words[:20]}")

print("\nCosine similarity between word pairs:")
word_pairs = [("dog", "cat"), ("cat", "play"), ("mat", "soft")]
for word1, word2 in word_pairs:
    try:
        vec1 = model.get_vector(word1)
        vec2 = model.get_vector(word2)

        # Normalize vectors
        vec1_norm = vec1 / (np.linalg.norm(vec1) + 1e-8)
        vec2_norm = vec2 / (np.linalg.norm(vec2) + 1e-8)

        # Cosine similarity
        similarity = np.dot(vec1_norm, vec2_norm)
        print(f"  '{word1}' vs '{word2}': {similarity:.4f}")
    except:
        pass



PART 7: UNDERSTANDING EMBEDDINGS

Sample vocabulary learned:
Total words: 25
Sample words: ['and', 'animals', 'are', 'cat', 'cats', 'clever', 'dog', 'dogs', 'friendly', 'i', 'independent', 'is', 'like', 'log', 'loyal', 'mat', 'my', 'on', 'play', 'sat']

Cosine similarity between word pairs:
  'dog' vs 'cat': 0.8801
  'cat' vs 'play': 0.7577
  'mat' vs 'soft': 0.7678
